# 02 — KG instantiation (validation & starter)

Validates that `kg/schema/ontology.ttl` actually works by using it: loads the
TBox, instantiates real POI data against it for a few sources (one per distinct
modelling pattern), runs SPARQL queries against the result, and leaves a
reusable helper + a fill-in-the-blanks section so the remaining sources can be
added the same way.

Three worked examples, chosen to cover the three distinct patterns from
`docs/kg_schema_design.md`:
1. **Museen** — the plain case: name, district, coordinates, address
2. **Parkanlagen** — adds `schema:amenityFeature` for boolean facts (dog-friendly,
   playground, water) plus a numeric `viennakg:areaSqm`
3. **Spielplätze** — the "separate nodes per feature" + equipment-list pattern

Everything else (Büchereien, Badestellen, Schwimmbäder, Sights, Wiener Linien
transport data) follows the same recipe — see the last section.

## Setup

In [1]:
import pandas as pd
import re
from rdflib import Graph, Namespace, URIRef, Literal, RDF
from rdflib.namespace import RDFS, XSD

In [2]:
VIENNAKG = Namespace("http://example.org/viennakg#")
SCHEMA = Namespace("https://schema.org/")
GEO = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")

g = Graph()
g.parse("../kg/schema/ontology.ttl", format="turtle")
g.bind("viennakg", VIENNAKG)
g.bind("schema", SCHEMA)
g.bind("geo", GEO)

print("Loaded TBox:", len(g), "triples")

Loaded TBox: 161 triples


## Shared helpers

`safe_id` turns any messy source ID (e.g. `"MUSEUMOGD.138686"`) into a URI-safe
local name. `parse_point` extracts the first `lon lat` pair out of a WKT `SHAPE`
string — same logic used throughout the EDA notebooks.

In [3]:
def safe_id(raw) -> str:
    return re.sub(r"[^A-Za-z0-9_]", "_", str(raw))

def parse_point(shape):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(shape))
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def bezirk_uri(bezirk):
    """BEZIRK int -> the matching viennakg:BezirkN individual, or None if missing/unmappable."""
    if pd.isnull(bezirk):
        return None
    return VIENNAKG[f"Bezirk{int(bezirk)}"]

## 1. Museen → `schema:Museum`

The plain case: one triple block per row for type, name, district, coordinates,
address, and an optional website.

In [4]:
museum = pd.read_csv("../data/processed/MUSEUMOGD_clean.csv")

for i, row in museum.iterrows():
    poi = VIENNAKG[f"museum_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.Museum))
    g.add((poi, SCHEMA.name, Literal(row["NAME"])))
    g.add((poi, SCHEMA.address, Literal(row["ADRESSE"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))
    if pd.notnull(row.get("WEITERE_INF")):
        g.add((poi, SCHEMA.url, Literal(row["WEITERE_INF"])))

print(f"Instantiated {len(museum)} Museum POIs. Graph size now: {len(g)} triples")

Instantiated 136 Museum POIs. Graph size now: 1111 triples


## 2. Parkanlagen → `schema:Park`

Adds the `schema:amenityFeature` pattern: each Ja/Nein column becomes its own
`schema:LocationFeatureSpecification` blank node with a `schema:name` and a
boolean `schema:value` — rather than three separate ad-hoc boolean properties.

In [5]:
from rdflib import BNode

parks = pd.read_csv("../data/processed/PARKINFOOGD_clean.csv")

AMENITY_MAP = {
    "SPIELEN_IM_PARK": "Playground",
    "WASSER_IM_PARK": "Water feature",
    "HUNDE_IM_PARK": "Dogs allowed",
}

for i, row in parks.iterrows():
    poi = VIENNAKG[f"park_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.Park))
    g.add((poi, SCHEMA.name, Literal(row["ANL_NAME"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))
    if pd.notnull(row.get("FLAECHE_M2")):
        g.add((poi, VIENNAKG.areaSqm, Literal(row["FLAECHE_M2"], datatype=XSD.decimal)))

    for col, label in AMENITY_MAP.items():
        val = str(row.get(col, "")).strip().lower() == "ja"
        feature = BNode()
        g.add((poi, SCHEMA.amenityFeature, feature))
        g.add((feature, RDF.type, SCHEMA.LocationFeatureSpecification))
        g.add((feature, SCHEMA.name, Literal(label)))
        g.add((feature, SCHEMA.value, Literal(val, datatype=XSD.boolean)))

print(f"Instantiated {len(parks)} Park POIs. Graph size now: {len(g)} triples")

Instantiated 1051 Park POIs. Graph size now: 20029 triples


## 3. Spielplätze → `viennakg:PlaygroundArea`

The "separate nodes per feature" pattern: every row is its own
`PlaygroundArea` instance (matches the source data — no aggregation by name).
`SPIELPLATZ_DETAIL`'s comma-separated equipment list becomes one
`LocationFeatureSpecification` per item via `schema:amenityFeature`, reusing the
exact same pattern as the Park amenities above.

In [6]:
playgrounds = pd.read_csv("../data/raw/SPIELPLATZPUNKTOGD.csv")

for i, row in playgrounds.iterrows():
    poi = VIENNAKG[f"playground_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, VIENNAKG.PlaygroundArea))
    g.add((poi, SCHEMA.name, Literal(row["ANL_NAME"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

    detail = row.get("SPIELPLATZ_DETAIL")
    if pd.notnull(detail):
        for item in [x.strip() for x in str(detail).split(",") if x.strip()]:
            feature = BNode()
            g.add((poi, SCHEMA.amenityFeature, feature))
            g.add((feature, RDF.type, SCHEMA.LocationFeatureSpecification))
            g.add((feature, SCHEMA.name, Literal(item)))
            g.add((feature, SCHEMA.value, Literal(True, datatype=XSD.boolean)))

print(f"Instantiated {len(playgrounds)} PlaygroundArea POIs. Graph size now: {len(g)} triples")

Instantiated 771 PlaygroundArea POIs. Graph size now: 40623 triples


## Validate with SPARQL

If the modelling actually works, these should return sensible results without
any special-casing per source — everything's queryable the same way regardless
of which of the three patterns produced it.

In [7]:
q = """
PREFIX schema: <https://schema.org/>
SELECT ?class (COUNT(?poi) AS ?n) WHERE {
    ?poi a ?class .
    FILTER(?class IN (schema:Museum, schema:Park, <http://example.org/viennakg#PlaygroundArea>))
}
GROUP BY ?class
"""
for row in g.query(q):
    print(row.n, g.qname(row["class"]))

136 schema:Museum
1051 schema:Park
771 viennakg:PlaygroundArea


In [8]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?name WHERE {
    ?poi a schema:Museum ;
         schema:containedInPlace viennakg:Bezirk1 ;
         schema:name ?name .
}
ORDER BY ?name
LIMIT 10
"""
print("Museums in Bezirk 1 (Innere Stadt), first 10:")
for row in g.query(q):
    print(" -", row.name)

Museums in Bezirk 1 (Innere Stadt), first 10:
 - 3D PicArt Museum
 - Albertina
 - Albertina modern
 - Beethoven Pasqualatihaus
 - Bezirksmuseum Innere Stadt
 - Dom Museum Wien
 - Ephesos-Museum
 - Esperantomuseum der Österreichischen Nationalbibliothek
 - Feuerwehrmuseum
 - Gemäldegalerie der Akademie der bildenden Künste


In [9]:
q = """
PREFIX schema: <https://schema.org/>
SELECT ?name WHERE {
    ?poi a schema:Park ;
         schema:name ?name ;
         schema:amenityFeature ?f .
    ?f schema:name "Dogs allowed" ;
       schema:value true .
}
LIMIT 10
"""
print("Dog-friendly parks, first 10:")
for row in g.query(q):
    print(" -", row.name)

Dog-friendly parks, first 10:
 - Erika-Morini-Park
 - PA Blériotgasse
 - Fridtjof-Nansen-Park
 - Ferdinand-Kaufmann-Platz
 - Andreas-Rett-Park
 - Prater - Rustenschacher
 - PA Donaustadtstraße
 - Vilma-Webenau-Park
 - GA Aspernstraße
 - PA Gaulgasse


## Save the demo graph

Writes the TBox + these three sources' instances out as one Turtle file — useful
to open in a Turtle-aware editor or load elsewhere to sanity-check by eye.

In [10]:
g.serialize(destination="../kg/instances_demo.ttl", format="turtle")
print("saved kg/instances_demo.ttl —", len(g), "triples total (TBox + 3 sources' ABox)")

saved kg/instances_demo.ttl — 40623 triples total (TBox + 3 sources' ABox)


## Your turn — remaining sources

Same recipe for everything else, following `docs/kg_schema_design.md`'s mapping
table. Copy one of the loops above and adjust:

| Source | Target class | Notes |
|---|---|---|
| `data/raw/BUECHEREIOGD.csv` | `schema.Library` | like Museum, but also has `OEFFNUNGSZEITEN1..6` → concatenate into `schema:openingHours`, plus `TELEFON`/`EMAIL` |
| `data/raw/BADESTELLENOGD.csv` | `VIENNAKG.BathingSite` | like Museum minus address (none in source); skip `BADEQUALITAET`/`TYP` per your call |
| `data/raw/SCHWIMMBADOGD.csv` | `VIENNAKG.SwimmingPool` | like Museum; `AUSLASTUNG_*` → `VIENNAKG.OccupancyStatus` instances is a stretch goal, not needed for a first pass |
| `data/processed/WIENTOURISMUS_sights_clean.csv` | `schema.TouristAttraction` | like Museum, but also add `VIENNAKG.hasCategory` → `VIENNAKG.subcat_Sehenswuerdigkeit` or `VIENNAKG.subcat_SchlossPalais` depending on `SUBCATEGORY_NAME` |
| `data/raw/wienerlinien-ogd-haltestellen.csv` + `-steige.csv` + `-linien.csv` | `VIENNAKG.Stop` / `VIENNAKG.Platform` / `VIENNAKG.Line` | three-way join on `FK_HALTESTELLEN_ID` / `FK_LINIEN_ID`, see `docs/wiener_linien_api_notes.md` for the join path |

Once all sources are in, this same SPARQL-querying approach (cell above) is how
the Reasoning Layer will query "POIs near a given stop, filtered by category and
live disruption status" — proximity itself isn't in the ontology as a static
edge, it'll be computed at query time from `geo:lat`/`geo:long`.

## 4. Büchereien → `schema:Library`

Like Museum, but also has `OEFFNUNGSZEITEN1..6` → concatenate into `schema:openingHours`, plus `TELEFON`/`EMAIL`.

In [ ]:
buechereien = pd.read_csv("../data/raw/BUECHEREIEN.csv")

for i, row in buechereien.iterrows():
    poi = VIENNAKG[f"buecherei_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, VIENNAKG.Library))
    g.add((poi, SCHEMA.name, Literal(row["ANL_NAME"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

    detail = row.get("BUECHEREI_DETAIL")
    if pd.notnull(detail):
        for item in [x.strip() for x in str(detail).split(",") if x.strip()]:
            feature = BNode()
            g.add((poi, SCHEMA.amenityFeature, feature))
            g.add((feature, RDF.type, SCHEMA.LocationFeatureSpecification))
            g.add((feature, SCHEMA.name, Literal(item)))
            g.add((feature, SCHEMA.value, Literal(True, datatype=XSD.boolean)))

print(f"Instantiated {len(buechereien)} Library POIs. Graph size now: {len(g)} triples")